# Scalar MHM-UNUSUAL reaction–diffusion

This notebook replays analytical and independent-code verification of the scalar method of Santiago, Valentin and Martins, [CILAMCE2025](https://doi.org/10.55592/cilamce2025.v5i.14270), Eqs.14–19. It uses archived results, not external comparison programs.

$$
Lu=\sigma u-\nabla\cdot(A\nabla u),\qquad
 a_{\mathrm{UN}}(u,v)=(A\nabla u,\nabla v)+(\sigma u,v)-\sum_T\tau_T(Lu,Lv)_T.
$$

The RHS includes the matching negative residual of the source. This is distinct from SUPG at zero advection. The raw physical flux is $q_h=-A\nabla u_h$; it is not claimed to be H(div)-conforming. The documentation states the inverse-constant and material-interface contracts.

The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/b17c01952899b1993ca0487e9dc9dc0c79f54addeaab4dbd0106a8e9f283e7ec/53_unusual-companion.zip"
COMPANION_SHA256 = "b17c01952899b1993ca0487e9dc9dc0c79f54addeaab4dbd0106a8e9f283e7ec"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/53_unusual.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Execute the analytical reaction layers through explicit equations

The current main uses the same analytical reaction–diffusion operator and mixed
boundary conditions as the complete analytical campaign:

$$
-\varepsilon\Delta p+p=1,\qquad
p|_{x=0,1}=0,\qquad
-\varepsilon\nabla p\cdot n|_{y=0,1}=0.
$$

The local continuous P1 pressure, P0 macroface multiplier and two local
subdivisions are fixed. Macro refinement is independent. Galerkin and UNUSUAL
use identical spaces and physical data, with strong nodal pressure on the
vertical exterior boundaries. `local_equations` declares the full strong
residual pairing, its negative test sign and those boundary constraints.
The exact exponential solution and its derivative are imported independently
of the element assembly. Absolute pressure, gradient, physical-flux and
energy errors use separate high-order integration; no maximum principle or
resolved-layer accuracy follows from the algebraic residual.

The variable anisotropic manufactured case additionally exercises the complete
diffusion divergence and declared continuum material/reaction bounds. The
`--study` recipe runs all five analytical levels and the separately declared
heterogeneous SPE10 physical study.



In [ ]:
import inspect
import numpy as np
from threadpoolctl import threadpool_limits
from pymhm import assemble, TriangleMesh, FaceSpace, SkeletonSpace
from examples.formulations.transport import (
    define_transport, local_equations, transport_constraints, recover_transport,
)
from examples.solve_unusual import configuration, errors

print(inspect.getsource(local_equations))
reaction_layers = []
with threadpool_limits(1):
    for epsilon in (1e-3, 1e-5):
        options, exact, exact_gradient = configuration("layer", epsilon)
        for method in ("galerkin", "unusual"):
            for n in (2, 4, 8):
                mesh = TriangleMesh.unit_square(n)
                skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(0) for _ in mesh.faces))
                natural = {
                    int(face): 0.0 for face in mesh.boundary_faces
                    if abs(mesh.normals[face, 1]) > 0.99
                }
                definition = define_transport(
                    mesh, skeleton=skeleton, degree=1, local_refinement=2,
                    stabilization=method, dirichlet_enforcement="strong",
                    neumann=natural, quadrature_order=10, **options,
                )
                system = assemble(definition.problem)
                solution = recover_transport(definition, system, system.solve(
                    fixed=definition.problem.fixed,
                    constraints=transport_constraints(definition, system),
                ))
                metrics = errors(solution, epsilon, exact, exact_gradient, 32)
                row = {"epsilon": epsilon, "method": method, "macro_n": n,
                       "residual": solution.hybrid.residual, **metrics}
                reaction_layers.append(row)
                print(row)
    options, exact, exact_gradient = configuration("tensor", 1.0)
    mesh = TriangleMesh.unit_square(2)
    skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))
    definition = define_transport(
        mesh, skeleton=skeleton, degree=2, local_refinement=2,
        stabilization="unusual", dirichlet_enforcement="strong",
        quadrature_order=10, **options,
    )
    system = assemble(definition.problem)
    solution = recover_transport(definition, system, system.solve(
        fixed=definition.problem.fixed,
        constraints=transport_constraints(definition, system),
    ))
    print("Variable anisotropic physical errors:", errors(
        solution, options["diffusion"], exact, exact_gradient, 12,
    ))


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib
    import json
    from pathlib import Path
    from IPython.display import Image, display

    DATA = ROOT / "examples/results/unusual"
    FIGURES = ROOT / "docs/figures/unusual"
    analytical = json.loads((DATA / "analytical.json").read_text())
    control = json.loads((DATA / "resolution-control.json").read_text())
    native = json.loads((DATA / "native-comparison.json").read_text())
    assert len(analytical["rows"]) == 40
    assert len(control["rows"]) == 3
    assert len(native["rows"]) == 10
    for record in (analytical, control):
        assert not record["source_changed_during_run"]
        for row in record["rows"]:
            if "archive" in row:
                assert hashlib.sha256((DATA / row["archive"]).read_bytes()).hexdigest() == row["archive_sha256"]
    print("Verified 43 analytical configurations and 10 native matched configurations.")

## Five-level analytical tests

The mixed-boundary exact problem has $f=\sigma=1$, $A=\epsilon I$, zero values at $x=0,1$ and zero normal flux at $y=0,1$.

$$
u(x,y)=1-\frac{e^{-x/\sqrt\epsilon}+e^{-(1-x)/\sqrt\epsilon}}{1+e^{-1/\sqrt\epsilon}}.
$$

P1/P0 with one red local refinement is tested at $n=2,4,8,16,32$. The tensor test uses $A=(1+x+y)[[2,0.3],[0.3,1]]$, $\sigma=3+x$, and exact $u=\sin(\pi x)\sin(\pi y)$ in P2/P1. Errors are integrated physical norms, not RMS values on display grids.

In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(FIGURES / "convergence.png")))

In [ ]:
if HISTORICAL_REPLAY:
    for row in analytical["rows"]:
        if row["n"] == 32 and row["method"] == "unusual":
            print(row["case"], "scalar L2 (%):", 100*row["scalar_relative_l2"],
                  "physical flux L2 (%):", 100*row["flux_relative_l2"])

## Boundary layers and resolution

A smaller maximum nodal error does not establish accuracy or a maximum principle. At epsilon=1e-5, UNUSUAL still has an 11.35% relative scalar error and approximately 88.21% relative flux error in the fixed P1/P0 spaces. The profiles preserve the one-sided macro traces.

The additional P3/P2 study changes both spaces explicitly while preserving the physical epsilon=0.001 problem. At n32 its relative flux error is 0.121856%, versus 23.2268% for P1/P0. These are distinct discretizations, not a relabeling of the conference figure.

In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(FIGURES / "profiles.png")))
    display(Image(filename=str(FIGURES / "resolved-layer-flux.png")))
    display(Image(filename=str(FIGURES / "tensor-flux.png")))

## Independent verification

DOLFINx/UFL tests compare full P1/P2/P3 matrices and loads, including derivatives of variable anisotropic diffusion. Separately, the supplied MHMUN-RAD_Parallel application executes in FreeFem++ 4.13 on five grids for each of P2/P0 and P3/P1. Its operator is unchanged; the comparison instruments precision and polynomial field output. The archived revision and source-access limitation are recorded below.

Agreement of matching discrete fields does not imply small discretization error. The trigonometric FreeFem application is distinct from the paper's analytical layer, and this notebook does not reproduce the SPE10 figure.

In [ ]:
if HISTORICAL_REPLAY:
    print(native["reference"])
    for metric in ("pressure_relative_difference", "flux_relative_difference"):
        print(metric, max(row[metric] for row in native["rows"]))
    display(Image(filename=str(FIGURES / "native-comparison.png")))

## Regenerating package-owned analytical data

```sh
python -m examples.solve_unusual
python -m examples.verify_unusual_resolution
python -m examples.plot_unusual
pixi run --locked -e fem python -m pytest -q tests/test_unusual_fenics.py
```

The release contains numerical evidence and provenance for the independent comparison; external solver sources and comparison execution programs are not included.

## Heterogeneous SPE10 reaction layers

The separate Section 4.2 study uses layer 36 Kx isotropically, reaction 1 and the original numerical coordinate/material units. The explicit source is zero; that heterogeneous subsection does not restate its value. This input choice is declared rather than identified with an unavailable historical file.

The nominal 128 macrotriangles, P1 local/P0 trace and r8/s8 resolution are retained. Uniform local/trace refinements and physical reaction-layer controls are different approximation spaces, labeled separately. The formulation, material and stabilization are unchanged. A separately assembled DOLFINx/UFL CG2 reference has final increments 0.11554% in pressure, 0.31754% in flux and 0.28015% in diffusion–reaction energy; it is not exact.

All differences below use physical intersections, two positive quadratures and the same reference denominator. Raw flux means $-K\nabla p$, not an H(div) reconstruction. Pressure extrema remain unmodified; small residuals do not establish a maximum principle.

The material-fitted trace control preserves the layer-resolved fine mesh and adds P0 moments at pixel crossings. This changes 32 uniform segments into 39–66 segments per macroface, without changing P1 or the element stabilization. Its remaining difference is measured against the same numerical reference.

At the same material-fitted trace, the c=.25 local control gives 4.1944% pressure, 5.3916% raw flux and 4.8988% physical energy differences. The two exact-overlay quadratures agree. Its pressure extrema remain −0.071375 and 1.435222; weak bottom moments close within 8e−15, but the incident corner values differ. These are unresolved physical traces, not clipped visualization values.


In [ ]:
if HISTORICAL_REPLAY:
    spe = ROOT / "examples/results/unusual-spe10"
    reference_name = "classical-cg2-graded-xy-1440x498"
    reference_record = json.loads((spe / f"{reference_name}.json").read_text())
    assert hashlib.sha256((spe / reference_record["archive"]).read_bytes()).hexdigest() == reference_record["archive_sha256"]
    for path in sorted(spe.glob("mhm-unusual-*-comparison.json")):
        record = json.loads(path.read_text())
        if len(record["rows"]) != 2:
            continue
        assert hashlib.sha256((spe / record["archive"]).read_bytes()).hexdigest() == record["archive_sha256"]
        assert record["reference_sha256"] == reference_record["archive_sha256"]
        values = record["rows"][-1]
        print(record["archive"], {key: 100 * values[key + "_relative"] for key in ("pressure", "flux", "energy")})

    physical = json.loads((spe / "physical-form-verification.json").read_text())
    assert len(physical["rows"]) == 8
    maximum_physical_residual = max(row["physical_free_relative_residual"] for row in physical["rows"])
    assert maximum_physical_residual < 1e-10
    print("Maximum reloaded CG2 physical residual:", maximum_physical_residual)
    replay = json.loads((spe / "consumer-replay-verification.json").read_text())
    assert len(replay["comparisons"]) == 9 and len(replay["classical"]) == 5
    for record in replay["comparisons"] + replay["classical"]:
        assert hashlib.sha256((spe / record["record"]).read_bytes()).hexdigest() == record["published_record_sha256"]


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("profiles", "resolution", "boundary-layer", "reaction-layer"):
        display(Image(filename=str(ROOT / "docs/figures/unusual-spe10" / f"{name}.png")))


The inlet maps expand the vertical display scale. Actual macro edges are overlaid; profiles retain both macro-side values and all fine-edge crossings. The [case page](https://ipes-lncc.github.io/pymhm/cases/unusual-spe10/) states the operator, source convention, spaces, geometric layer criterion and complete provenance. Package-owned acquisition and replay commands are separate from this lightweight notebook.
